In [1]:
import gymnasium as gym
import numpy as np
import subprocess
import threading
import os
import wave
# from sbx import DDPG, DQN, PPO, SAC, TD3, TQC, CrossQ
from pathlib import Path
from gymnasium.wrappers import RecordVideo
from sdlarch_rl import make
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import DummyVecEnv, VecVideoRecorder, VecFrameStack, VecNormalize, VecTransposeImage
from sdlarch_rl.utils.discretizer import MainDiscretizer
from stable_baselines3.common.atari_wrappers import WarpFrame
from stable_baselines3.common.vec_env import VecMonitor
from stable_baselines3 import PPO
from sdlarch_rl.utils.utils import get_latest_model, TrainAndLoggingCallback
import pygame
import cv2

import numpy as np

LOAD_FROM="./model-nsm/"
GAME = "NewSuperMarioBros-Wii"
BEST_MODEL = "best_model_5000"
LOAD_FROM_VIDEO= LOAD_FROM + BEST_MODEL
frames = []
#STATE="YoshiIsland1"

class SkipFrameWithAudioAndVideo(gym.Wrapper):
    def __init__(self, env, skip, my_env, data, frames):
        super().__init__(env)
        self._skip = skip
        self.my_env = my_env
        self.frames = frames

        self.data = data


    def step(self, action):
        total_reward = 0.0
        done = False
        for i in range(self._skip):
            frame, reward, terminated, trunk, info = self.env.step(action)

            sound = my_env.em.get_audio()
            self.data['audio_data'] += sound.tobytes()
            self.frames.append(frame)
            
            total_reward += reward
            if terminated:
                break
        return frame, total_reward, terminated, trunk, info

os.makedirs("videos", exist_ok=True)

combos = [
    [],
    # run
    ["LEFT", "X"],
    ["RIGHT", "X"],
    
    # jump and run
    ["Y"],
    ["LEFT", "X", "Y"],
    ["RIGHT", "X", "Y"],
    
    #shake and run
    ["R2"],
    ["LEFT", "X", "R2"],
    ["RIGHT", "X", "R2"],
]

video_filename = "videos/video-episode-0.mp4"
audio_filename = "videos/game_audio.wav"

audio_data = b""
arate = None
framerate = None

data = {
    "audio_data": audio_data,
    "arate": arate,
    "framerate": framerate,
}

        
def make_env():
    def _init():
        global my_env
        
        my_env = make(
            GAME, 
            env_id=15,
        )
        my_env.set_buttons(["B", "Y", "SELECT", "START", "LEFT", "RIGHT", "DOWN", "UP","A", "X", "L1", "R1", "L2", "R2", "L3", "R3"])

        env = MainDiscretizer(
            my_env,
            combos,
        )

        env = SkipFrameWithAudioAndVideo(env, skip=4, my_env=my_env, data=data, frames=frames)
        env = WarpFrame(env, width=96, height=96)
        
    
        return env
    return _init


latest_model_path = LOAD_FROM_VIDEO
video_prefix = BEST_MODEL + ".mp4"

env = make_vec_env(make_env(), n_envs=1, vec_env_cls=DummyVecEnv)
env = VecFrameStack(env, 4, channels_order='last')
vec_env = VecMonitor(env)

render_mode = "rgb_array"

model = PPO.load(
# model = RecurrentPPO.load(
    str(latest_model_path), 
    env=env, 
    verbose=0,
)

obs = env.reset()
done = False

data['arate'] = my_env.unwrapped.em.get_audio_rate()
data['framerate'] = my_env.unwrapped.em.get_frame_rate()


# data['arate'] = 48000
# data['framerate']= 59.998
print("arate: ", data['arate'])
print("framerate: ", data['framerate'])

dones = 0

frameskip = 4

pygame.init()

SCREEN_WIDTH = 800
SCREEN_HEIGHT = 640
window = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
clock = pygame.time.Clock()

count = 0
while True:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            done = True
    env.render()
    action, _ = model.predict(obs, deterministic=False)

    obs, reward, done, info = env.step(action)

    # data['arate'] = env.unwrapped.em.get_audio_rate()
    # data['framerate'] = env.unwrapped.em.get_frame_rate()
    # sound = env.unwrapped.em.get_audio()
    # data['audio_data'] += sound.tobytes()

    count += 1

    stack = obs[0]  # shape: (128, 128, 4)
    # frames = np.split(stack, 4, axis=2)
    #frames = np.split(stack, 1, axis=2)

    gray_frame = stack[..., -1]  # (128, 128)
    # print(gray_frame.shape)
    gray_frame_rgb = cv2.cvtColor(gray_frame, cv2.COLOR_GRAY2RGB)  # (128, 128, 3)
    
    obs_resized = cv2.resize(gray_frame_rgb, (SCREEN_WIDTH, SCREEN_HEIGHT))  # (H, W, 3)
    
    obs_surface = pygame.surfarray.make_surface(np.transpose(obs_resized, (1, 0, 2)))
    window.blit(obs_surface, (0, 0))
    pygame.display.update()

    if done:
        break

pygame.quit()
env.close()

out = cv2.VideoWriter(video_filename, cv2.VideoWriter_fourcc(*"mp4v"), 60, (640, 480))
for f in frames:
    out.write(cv2.cvtColor(f, cv2.COLOR_RGB2BGR))
out.release()

print("arate: ", data['arate'])
print("framerate: ", data['framerate'])

# save audio
with wave.open(audio_filename, "wb") as wf:
    wf.setnchannels(2)  # Mono
    wf.setsampwidth(2)  # 16-bit PCM
    wf.setframerate(data['arate'])  # Audio rate
    wf.writeframes(data['audio_data'])

ffmpeg_cmd = [
    "ffmpeg", "-y",
    "-r", str(data['framerate']),
    "-i", video_filename,
    "-i", audio_filename,
    # "-vf", "scale=1280:720", # hd resolution
    "-vf", f"scale={SCREEN_WIDTH}:{SCREEN_HEIGHT},setdar=16:9,setsar=1", # HD
    # "-vf", "scale=854:480,setdar=16:9,setsar=1", # 16x9
    "-c:v", "libx264",
    # "-preset", "veryslow",
     "-crf", "10", # quality 10 ~ 50 (10 is better)
    "-c:a", "aac",
    "-b:a", "128k",
    "-ac","2",
    "-map", "0:v:0",       
    "-map", "1:a:0",
    "-strict", "experimental",
    "-shortest",
    "./videos/" + video_prefix
]

subprocess.run(ffmpeg_cmd)

print("✅ Finished vídeos")

D:\Python311\Lib\site-packages\pygame\pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists


Detected game: newsupermariobros-wii
Starting dolphin core... D:\projects\sdlarch\sdlarch_rl\./cores/dolphin/dolphin_libretro.dll D:\projects\sdlarch\sdlarch_rl\roms\NewSuperMarioBros-Wii\rom.rvz 15
Variable: dolphin_renderer = Hardware

Variable: dolphin_efb_scale = x1 (640 x 528)

Variable: dolphin_widescreen = enabled

Variable: dolphin_widescreen_hack = disabled

Variable: dolphin_shader_compilation_mode = a-sync Skip Rendering

Variable: dolphin_wait_for_shaders = disabled

Variable: dolphin_progressive_scan = enabled

Variable: dolphin_pal60 = enabled

Variable: dolphin_anti_aliasing = None

Variable: dolphin_max_anisotropy = 1x

Variable: dolphin_skip_dupe_frames = enabled

Variable: dolphin_immediate_xfb = disabled

Variable: dolphin_efb_scaled_copy = enabled

Variable: dolphin_force_texture_filtering = disabled

Variable: dolphin_efb_to_texture = enabled

Variable: dolphin_texture_cache_accuracy = Fast

Variable: dolphin_gpu_texture_decoding = enabled

Variable: dolphin_fast_d

D:\Python311\Lib\site-packages\stable_baselines3\common\vec_env\vec_monitor.py:44: UserWarning: The environment is already wrapped with a `Monitor` wrapperbut you are wrapping it with a `VecMonitor` wrapper, the `Monitor` statistics will beoverwritten by the `VecMonitor` ones.
  warnings.warn(


arate:  32000.0
framerate:  59.94005584716797
[[[184   0 237]
  [184   0 237]
  [184   0 237]
  ...
  [184   0 237]
  [184   0 237]
  [184   0 237]]

 [[184   0 237]
  [184   0 237]
  [184   0 237]
  ...
  [184   0 237]
  [184   0 237]
  [184   0 237]]

 [[184   0 237]
  [184   0 237]
  [184   0 237]
  ...
  [184   0 237]
  [184   0 237]
  [184   0 237]]

 ...

 [[184   0 237]
  [184   0 237]
  [184   0 237]
  ...
  [184   0 237]
  [184   0 237]
  [184   0 237]]

 [[184   0 237]
  [184   0 237]
  [184   0 237]
  ...
  [184   0 237]
  [184   0 237]
  [184   0 237]]

 [[184   0 237]
  [184   0 237]
  [184   0 237]
  ...
  [184   0 237]
  [184   0 237]
  [184   0 237]]]
[[[100 200 255]
  [100 200 255]
  [100 200 255]
  ...
  [100 200 255]
  [100 200 255]
  [100 200 255]]

 [[100 200 255]
  [100 200 255]
  [100 200 255]
  ...
  [100 200 255]
  [100 200 255]
  [100 200 255]]

 [[100 200 255]
  [101 200 255]
  [100 200 255]
  ...
  [100 200 255]
  [100 200 255]
  [100 200 255]]

 ...

 [[214